## Trying to perfect the data

Previously after cleaning the data , i realised that for our specific outcome the data we have is not perfect , because i am trying to predict the startups status after 5 years , so i need data of only those companies who have been operating for more than 5 years . After sorting it in a such i way we get only 14,259 were funded early enough (before 2010) to have a known 5-year outcome
Of those, 14,011 have a usable status (dropped 248 with no status) 

But here's the real problem to notice: the split is 89% success vs 11% fail (12,477 vs 1,534). That's badly imbalanced.

But still we are going forward with the project and hope to handle the imbalance using xgboost scale_pos_weight

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

df=pd.read_csv("/home/student/Projects/Startup-outcome-intelligence/data/processed/startup_data_prepared.csv")
df.columns = df.columns.str.strip()



In [2]:
df.head()

,category_list,market,funding_total_usd,status,country_code,funding_rounds,founded_at,first_funding_at,last_funding_at,equity_crowdfunding,...,category_missing,country_missing,funding_total_missing,Startup_age,days_to_funding,funding_duration_days,funding_per_round,single_funding_event,funding_stage_reached,has_institutional_vc
0,|Entertainment|Politics|Social Media|News|,News,1750000.0,acquired,USA,1.0,2012-06-01,2012-06-30,2012-06-30,0.0,...,0,0,0,3.0,29.0,0.0,1750000.0,1,1,0
1,|Games|,Games,4000000.0,operating,USA,2.0,NaN,2010-06-04,2010-09-23,0.0,...,0,0,0,NaN,NaN,111.0,2000000.0,0,0,1
2,|Publishing|Education|,Publishing,40000.0,operating,EST,1.0,2012-10-26,2012-08-09,2012-08-09,0.0,...,0,0,0,3.0,0.0,0.0,40000.0,1,1,0
3,|Electronics|Guides|Coffee|Restaurants|Music|i...,Electronics,1500000.0,operating,GBR,1.0,2011-04-01,2011-04-01,2011-04-01,0.0,...,0,0,0,4.0,0.0,0.0,1500000.0,1,1,0
4,|Tourism|Entertainment|Games|,Tourism,60000.0,operating,USA,2.0,2014-01-01,2014-08-17,2014-09-26,60000.0,...,0,0,0,1.0,228.0,40.0,30000.0,0,0,0


# Filtering 
filtering the companies which were funded before 2010 cse the dataset is a snapshot from 2015 and we need a 5 year window  and deleting companies with status NULL  

In [3]:
df = df[df["first_funding_at"].notna()]
df=df.drop(df[df["first_funding_at"]>="2010-01-01"].index)

In [4]:
df.shape

(13921, 27)

In [5]:
df=df[df["status"].notna()]

In [6]:
df.head(50)

,category_list,market,funding_total_usd,status,country_code,funding_rounds,founded_at,first_funding_at,last_funding_at,equity_crowdfunding,...,category_missing,country_missing,funding_total_missing,Startup_age,days_to_funding,funding_duration_days,funding_per_round,single_funding_event,funding_stage_reached,has_institutional_vc
6,|Advertising|,Advertising,4912393.0,closed,ARG,1.0,NaN,2007-01-16,2007-01-16,0.0,...,0,0,0,NaN,NaN,0.0,4.912393e+06,1,0,0
7,|Curated Web|,Curated Web,2000000.0,operating,missing,1.0,2007-01-01,2008-03-19,2008-03-19,0.0,...,0,1,0,8.0,443.0,0.0,2.000000e+06,1,2,1
16,|E-Commerce|,E-Commerce,2050000.0,operating,USA,4.0,2001-01-01,2009-06-18,2011-12-28,0.0,...,0,0,0,14.0,3090.0,923.0,5.125000e+05,0,0,0
18,|Marketplaces|Art|E-Commerce|,Marketplaces,500000.0,acquired,missing,1.0,2009-01-01,2009-05-15,2009-05-15,0.0,...,0,1,0,6.0,134.0,0.0,5.000000e+05,1,1,0
21,|Curated Web|,Curated Web,4962651.0,operating,USA,6.0,2008-01-01,2009-10-14,2014-09-19,0.0,...,0,0,0,7.0,652.0,1801.0,8.271085e+05,0,0,1
24,|Hospitality|,Hospitality,3000000.0,operating,missing,2.0,NaN,2008-01-07,2010-08-13,0.0,...,0,1,0,NaN,NaN,949.0,1.500000e+06,0,2,1
25,|Education|,Education,4500000.0,operating,CHN,2.0,NaN,2006-01-01,2006-09-01,0.0,...,0,0,0,NaN,NaN,243.0,2.250000e+06,0,0,1
30,|Curated Web|,Curated Web,50000.0,closed,USA,1.0,2009-04-01,2009-04-01,2009-04-01,0.0,...,0,0,0,6.0,0.0,0.0,5.000000e+04,1,1,0
51,|Manufacturing|,Manufacturing,66450000.0,operating,USA,8.0,2007-01-01,2008-03-27,2013-12-12,0.0,...,0,0,0,8.0,451.0,2086.0,8.306250e+06,0,3,1
52,|E-Commerce|,E-Commerce,2000000.0,operating,CHN,1.0,NaN,2007-02-01,2007-02-01,0.0,...,0,0,0,NaN,NaN,0.0,2.000000e+06,1,0,0


In [7]:
df.shape

(13677, 27)

In [8]:
df["status"].value_counts()

status
operating    9676
acquired     2517
closed       1484
Name: count, dtype: int64

# Creating labels for closed=0 and any form of existance as 1 

In [9]:
df["label"]=(df["status"]!="closed").astype(int)

In [10]:
df.head(20)

,category_list,market,funding_total_usd,status,country_code,funding_rounds,founded_at,first_funding_at,last_funding_at,equity_crowdfunding,...,country_missing,funding_total_missing,Startup_age,days_to_funding,funding_duration_days,funding_per_round,single_funding_event,funding_stage_reached,has_institutional_vc,label
6,|Advertising|,Advertising,4912393.0,closed,ARG,1.0,NaN,2007-01-16,2007-01-16,0.0,...,0,0,NaN,NaN,0.0,4.912393e+06,1,0,0,0
7,|Curated Web|,Curated Web,2000000.0,operating,missing,1.0,2007-01-01,2008-03-19,2008-03-19,0.0,...,1,0,8.0,443.0,0.0,2.000000e+06,1,2,1,1
16,|E-Commerce|,E-Commerce,2050000.0,operating,USA,4.0,2001-01-01,2009-06-18,2011-12-28,0.0,...,0,0,14.0,3090.0,923.0,5.125000e+05,0,0,0,1
18,|Marketplaces|Art|E-Commerce|,Marketplaces,500000.0,acquired,missing,1.0,2009-01-01,2009-05-15,2009-05-15,0.0,...,1,0,6.0,134.0,0.0,5.000000e+05,1,1,0,1
21,|Curated Web|,Curated Web,4962651.0,operating,USA,6.0,2008-01-01,2009-10-14,2014-09-19,0.0,...,0,0,7.0,652.0,1801.0,8.271085e+05,0,0,1,1
24,|Hospitality|,Hospitality,3000000.0,operating,missing,2.0,NaN,2008-01-07,2010-08-13,0.0,...,1,0,NaN,NaN,949.0,1.500000e+06,0,2,1,1
25,|Education|,Education,4500000.0,operating,CHN,2.0,NaN,2006-01-01,2006-09-01,0.0,...,0,0,NaN,NaN,243.0,2.250000e+06,0,0,1,1
30,|Curated Web|,Curated Web,50000.0,closed,USA,1.0,2009-04-01,2009-04-01,2009-04-01,0.0,...,0,0,6.0,0.0,0.0,5.000000e+04,1,1,0,0
51,|Manufacturing|,Manufacturing,66450000.0,operating,USA,8.0,2007-01-01,2008-03-27,2013-12-12,0.0,...,0,0,8.0,451.0,2086.0,8.306250e+06,0,3,1,1
52,|E-Commerce|,E-Commerce,2000000.0,operating,CHN,1.0,NaN,2007-02-01,2007-02-01,0.0,...,0,0,NaN,NaN,0.0,2.000000e+06,1,0,0,1


In [11]:
df["label"].value_counts()

label
1    12193
0     1484
Name: count, dtype: int64

# Extracting few more info before dropping year values

In [12]:
df["founded_at"] = pd.to_datetime(df["founded_at"])
df["founded_year"] = df["founded_at"].dt.year

# Further dropping useless features after deriving required stuff

In [13]:
df = df.drop(columns=["status","founded_at", "first_funding_at", "last_funding_at"])

In [14]:
df.head()

,category_list,market,funding_total_usd,country_code,funding_rounds,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,...,funding_total_missing,Startup_age,days_to_funding,funding_duration_days,funding_per_round,single_funding_event,funding_stage_reached,has_institutional_vc,label,founded_year
6,|Advertising|,Advertising,4912393.0,ARG,1.0,0.0,4912393.0,0.0,0.0,0.0,...,0,NaN,NaN,0.0,4912393.0,1,0,0,0,NaN
7,|Curated Web|,Curated Web,2000000.0,missing,1.0,0.0,0.0,0.0,0.0,0.0,...,0,8.0,443.0,0.0,2000000.0,1,2,1,1,2007.0
16,|E-Commerce|,E-Commerce,2050000.0,USA,4.0,0.0,0.0,0.0,2050000.0,0.0,...,0,14.0,3090.0,923.0,512500.0,0,0,0,1,2001.0
18,|Marketplaces|Art|E-Commerce|,Marketplaces,500000.0,missing,1.0,0.0,0.0,0.0,0.0,0.0,...,0,6.0,134.0,0.0,500000.0,1,1,0,1,2009.0
21,|Curated Web|,Curated Web,4962651.0,USA,6.0,0.0,0.0,0.0,1147879.0,0.0,...,0,7.0,652.0,1801.0,827108.5,0,0,1,1,2008.0


## Going to encode 

Categorical values need to be encoded 

First we will understand the nature of categorical values , then decide what to do 

# COUNTRY CODE :
while looking at count we see that many have only one instance which is not worth keeping for our dataset  , so we will only keep countries with at least 30 instances and club the rest

In [15]:
df["country_code"].value_counts()

country_code
USA        8647
missing    1073
GBR         744
CHN         508
FRA         359
           ... 
UZB           1
KWT           1
MKD           1
MAF           1
KEN           1
Name: count, Length: 86, dtype: int64

In [16]:
count=df["country_code"].value_counts()
keep=count[count>=30].index

df["country_code"]=df["country_code"].where(df["country_code"].isin(keep),"other")
df["country_code"].value_counts()

country_code
USA        8647
missing    1073
GBR         744
CHN         508
other       370
FRA         359
CAN         347
DEU         290
ISR         249
IND         150
ESP         137
SWE         124
NLD          83
IRL          78
CHE          66
DNK          64
AUS          60
JPN          58
BEL          56
SGP          53
ITA          46
FIN          40
BRA          39
NOR          36
Name: count, dtype: int64

# Market:
we again keep markets with more than 30 instances and club others

In [17]:
df["market"].value_counts().head(100)

market
 Software                   1746
 Biotechnology              1211
missing                      672
 Curated Web                 585
 Mobile                      545
                            ... 
 Chat                         12
 Information Technology       12
 Photo Sharing                12
 Virtual Worlds               11
 Energy                       11
Name: count, Length: 100, dtype: int64

In [18]:
#i saw a leading whitespace so i will strip
df["market"]=df["market"].str.strip()

In [19]:
count=df["market"].value_counts()
keep=count[count>=30].index

df["market"]=df["market"].where(df["market"].isin(keep),"other")
df["market"].value_counts()

market
other                   2376
Software                1746
Biotechnology           1211
missing                  672
Curated Web              585
Mobile                   545
Clean Technology         510
Enterprise Software      502
Hardware + Software      438
Games                    422
Advertising              400
Health Care              379
Semiconductors           342
E-Commerce               303
Security                 247
Web Hosting              246
Social Media             235
Finance                  208
Manufacturing            189
Health and Wellness      166
Analytics                150
Messaging                137
Consulting               132
Search                   126
Education                121
News                     107
Music                    100
Travel                    83
Video                     80
Public Relations          78
Technology                78
SaaS                      62
Photography               60
Real Estate               59
Interne

# Category: 
we see lot of category and market values are overlapping so i would prefer to drop it cse its mostly redundant with market values 

In [20]:
df["category_list"].value_counts().head(50)

category_list
|Software|                             1443
|Biotechnology|                        1178
missing                                 672
|Clean Technology|                      467
|Hardware + Software|                   405
|Curated Web|                           401
|Mobile|                                378
|Enterprise Software|                   371
|Semiconductors|                        326
|Games|                                 294
|Health Care|                           281
|Advertising|                           260
|Web Hosting|                           216
|E-Commerce|                            209
|Security|                              204
|Manufacturing|                         177
|Finance|                               158
|Health and Wellness|                   138
|Social Media|                          115
|Consulting|                            105
|Messaging|                              93
|Analytics|                              92
|Education|       

In [21]:
df=df.drop(columns="category_list")

In [22]:
df.head(50)

,market,funding_total_usd,country_code,funding_rounds,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,...,funding_total_missing,Startup_age,days_to_funding,funding_duration_days,funding_per_round,single_funding_event,funding_stage_reached,has_institutional_vc,label,founded_year
6,Advertising,4912393.0,other,1.0,0.0,4912393.0,0.0,0.0,0.0,0.0,...,0,NaN,NaN,0.0,4.912393e+06,1,0,0,0,NaN
7,Curated Web,2000000.0,missing,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,8.0,443.0,0.0,2.000000e+06,1,2,1,1,2007.0
16,E-Commerce,2050000.0,USA,4.0,0.0,0.0,0.0,2050000.0,0.0,0.0,...,0,14.0,3090.0,923.0,5.125000e+05,0,0,0,1,2001.0
18,other,500000.0,missing,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,6.0,134.0,0.0,5.000000e+05,1,1,0,1,2009.0
21,Curated Web,4962651.0,USA,6.0,0.0,0.0,0.0,1147879.0,0.0,0.0,...,0,7.0,652.0,1801.0,8.271085e+05,0,0,1,1,2008.0
24,Hospitality,3000000.0,missing,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,NaN,NaN,949.0,1.500000e+06,0,2,1,1,NaN
25,Education,4500000.0,CHN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,NaN,NaN,243.0,2.250000e+06,0,0,1,1,NaN
30,Curated Web,50000.0,USA,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,6.0,0.0,0.0,5.000000e+04,1,1,0,0,2009.0
51,Manufacturing,66450000.0,USA,8.0,0.0,0.0,0.0,0.0,0.0,3000000.0,...,0,8.0,451.0,2086.0,8.306250e+06,0,3,1,1,2007.0
52,E-Commerce,2000000.0,CHN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,NaN,NaN,0.0,2.000000e+06,1,0,0,1,NaN


## ENCODING 

now since i have sorted the categorical values  , i will encode them using get_dummies 

In [23]:
df=pd.get_dummies(df,columns=["market","country_code"],dtype=int)

In [24]:
df.head()

,funding_total_usd,funding_rounds,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,product_crowdfunding,market_mising,...,country_code_ISR,country_code_ITA,country_code_JPN,country_code_NLD,country_code_NOR,country_code_SGP,country_code_SWE,country_code_USA,country_code_missing,country_code_other
6,4912393.0,1.0,0.0,4912393.0,0.0,0.0,0.0,0.0,0.0,0,...,0,0,0,0,0,0,0,0,0,1
7,2000000.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0,0,0,0,0,0,0,0,1,0
16,2050000.0,4.0,0.0,0.0,0.0,2050000.0,0.0,0.0,0.0,0,...,0,0,0,0,0,0,0,1,0,0
18,500000.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,...,0,0,0,0,0,0,0,0,1,0
21,4962651.0,6.0,0.0,0.0,0.0,1147879.0,0.0,0.0,0.0,0,...,0,0,0,0,0,0,0,1,0,0


## CHECKING
finally checking if i forgot something

In [25]:
df.isna().sum()

funding_total_usd       0
funding_rounds          0
equity_crowdfunding     0
undisclosed             0
convertible_note        0
                       ..
country_code_SGP        0
country_code_SWE        0
country_code_USA        0
country_code_missing    0
country_code_other      0
Length: 94, dtype: int64

In [26]:
df.dtypes

funding_total_usd       float64
funding_rounds          float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
                         ...   
country_code_SGP          int64
country_code_SWE          int64
country_code_USA          int64
country_code_missing      int64
country_code_other        int64
Length: 94, dtype: object

In [27]:
df.shape

(13677, 94)

In [29]:
df.describe()

,funding_total_usd,funding_rounds,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,product_crowdfunding,market_mising,...,country_code_ISR,country_code_ITA,country_code_JPN,country_code_NLD,country_code_NOR,country_code_SGP,country_code_SWE,country_code_USA,country_code_missing,country_code_other
count,1.367700e+04,13677.000000,1.367700e+04,1.367700e+04,1.367700e+04,1.367700e+04,1.367700e+04,1.367700e+04,1.367700e+04,13677.000000,...,13677.000000,13677.000000,13677.000000,13677.000000,13677.000000,13677.000000,13677.000000,13677.000000,13677.000000,13677.000000
mean,2.303830e+07,2.210719,5.238802e+02,1.958440e+05,2.731970e+04,1.715047e+06,8.033180e+04,1.236281e+05,5.543944e+03,0.049134,...,0.018206,0.003363,0.004241,0.006069,0.002632,0.003875,0.009066,0.632229,0.078453,0.027053
std,9.701114e+07,1.778226,3.877430e+04,3.503576e+06,7.255754e+05,2.370995e+07,5.735705e+05,3.267106e+06,6.160104e+05,0.216155,...,0.133700,0.057899,0.064985,0.077667,0.051239,0.062132,0.094788,0.482216,0.268893,0.162243
min,1.000000e+03,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000e+06,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5.000000e+06,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,1.902641e+07,3.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
max,5.700000e+09,18.000000,4.182382e+06,2.594178e+08,6.000000e+07,1.500000e+09,3.025439e+07,2.518600e+08,7.200000e+07,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## BINGO

finally finalising my dataset and marking end of all cleaning (i hope so)

In [35]:
df.to_csv("../data/processed/startup_data_final.csv",index=False)